# A2-Lite 640×192 — paired accuracy audit
Latest revision: **A2-LITE-640-PAIRED-ACCURACY-2026-10-12-r1**

Compare two frozen ONNX graphs on **all 3,769 Chen validation images**: full-resolution A2-Lite 1280×384 (control), and the SAME E1/D1 weights at 640×192 (phone-speed candidate). No training, distillation, native source clone, checkpoint loading, old virtual environment, CUDA extension build or Core ML. A CPU runtime is sufficient; a GPU runtime also works but the GPU is unused.

Measure product-taxonomy Vehicle/Pedestrian 3D and BEV AP_R40, plus existing nearby recall: 2D IoU≥0.5, serialized score≥0.001, GT range Vehicle<40 m/Pedestrian<30 m. Nearby recall is NOT proof of correct 3D geometry or product safety. The native 2-decimal KITTI text format, top-50 class/query selection, class threshold 0.001 and meanshape=False are held fixed. Invalid geometry is retained in raw predictions and counted with zero-volume dimensions for 3D/BEV AP—not silently dropped.

The phone 640×192 run passed the configured model-only gate (44.6 ms p95, 19.9 FPS), but sustained p95 was 52.5 ms and the camera pipeline remains untested. This notebook does not repeat phone timing or authorize deployment/training. It measures the accuracy gap first. Full A2 historical metrics are context only, not a newly paired pruning ablation.

Run sections 1–5 in order. Only validation data is copied to a dedicated local folder, reducing repeated Drive I/O. Interrupted inference is resumable: rerun section 4 with the same unchanged RUN_ID; after a session reset, rerun sections 1–3 first. Use a NEW RUN_ID if code, CPU package versions, model or data identity changed. Never delete old results to bypass checks. Return a2_lite_accuracy_results.zip.

## 1. Mount Drive, set paths and define every helper
Use the existing export folders on Drive. The original downloaded ZIPs also work for BASELINE and COMPACT. Neither initialization checkpoints nor old runtimes are needed.

In [ ]:
# A2-LITE-640-PAIRED-ACCURACY-2026-10-12-r1
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from datetime import datetime, timezone
from collections import deque
import json, re, shlex, subprocess, sys
REVISION='A2-LITE-640-PAIRED-ACCURACY-2026-10-12-r1'
RUN_ID='a2_lite_640_accuracy_r1'
if not re.fullmatch(r'[a-z0-9_]+',RUN_ID): raise ValueError('Use lowercase letters, digits and underscores')
MOBILE_REPO=Path('/content/mobile_adas3d')
BASELINE=Path('/content/drive/MyDrive/mobile_adas3d_outputs/edge_candidates/a2_lite/a2_lite_e1_d1_cpu_screen_r1')
COMPACT=Path('/content/drive/MyDrive/mobile_adas3d_outputs/edge_candidates/a2_lite_resolution/a2_lite_resolution_cpu_screen_r1')
SPLITS=Path('/content/drive/MyDrive/mobile_adas3d_splits/kitti_chen')
DATASET=Path('/content/kitti_a2_lite_accuracy_'+RUN_ID)
CANDIDATES=[Path(p) for p in ('/content/kitti_m68','/content/kitti_m67','/content/kitti_m66','/content/kitti_m65','/content/kitti','/content/monodetr_kitti_a2','/content/drive/MyDrive/datasets/kitti')]
OUTPUT=Path('/content/drive/MyDrive/mobile_adas3d_outputs/edge_candidates/a2_lite_accuracy')/RUN_ID
PYTHON=Path(sys.executable); SCRIPT=MOBILE_REPO/'scripts/evaluate_a2_lite_accuracy.py'
LOGS=OUTPUT/'colab_logs'; LOGS.mkdir(parents=True,exist_ok=True); READY=False
def run_logged(command,name,cwd=None):
    command=list(map(str,command))
    path=LOGS/f"{name}_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')}.log"
    print('+',shlex.join(command),'\nDurable log:',path,flush=True); tail=deque(maxlen=40)
    with path.open('w') as log:
        log.write(shlex.join(command)+'\n'); log.flush()
        process=subprocess.Popen(command,cwd=str(cwd) if cwd else None,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        try:
            for line in process.stdout:
                print(line,end='',flush=True); log.write(line); log.flush(); tail.append(line.rstrip())
        except BaseException:
            process.terminate()
            try: process.wait(timeout=10)
            except subprocess.TimeoutExpired: process.kill(); process.wait()
            raise
        finally: process.stdout.close()
        code=process.wait(); log.write(f'\nEXIT CODE: {code}\n'); log.flush()
    if code: raise RuntimeError(f'Exit {code}; full log={path}\n'+'\n'.join(tail))
    return path
def stage(name,*extra):
    if not READY: raise RuntimeError('Run sections 1 and 2 first')
    return run_logged([PYTHON,'-u',SCRIPT,name,'--output',OUTPUT,*extra],name,MOBILE_REPO)
print(REVISION,'\nOutput:',OUTPUT,'\nEvaluation only. No training or GPU build.')


## 2. Update clean main and install CPU dependencies
No Torch/CUDA replacement or old runtime restore. CPU package identities are recorded; unrelated GPU allocation is not part of resume identity.

In [ ]:
READY=False
if not MOBILE_REPO.exists(): run_logged(['git','clone','--branch','main','https://github.com/Ali-RT/mobile_adas3d.git',MOBILE_REPO],'clone')
dirty=subprocess.check_output(['git','status','--porcelain','--untracked-files=no'],cwd=MOBILE_REPO,text=True).strip()
if dirty: raise RuntimeError('Preserve tracked edits and use a clean main checkout.\n'+dirty)
branch=subprocess.check_output(['git','branch','--show-current'],cwd=MOBILE_REPO,text=True).strip()
if branch!='main': raise RuntimeError(f'Use main, not {branch!r}')
run_logged(['git','pull','--ff-only','origin','main'],'update',MOBILE_REPO)
if not SCRIPT.is_file(): raise FileNotFoundError(SCRIPT)
for path in (BASELINE,COMPACT,SPLITS/'val.txt'):
    if not path.exists(): raise FileNotFoundError(f'{path}: correct the path in section 1')
run_logged([PYTHON,'-m','pip','install','-q','timm==1.0.27','numpy==2.1.3','onnxruntime==1.30.0','pillow','opencv-python-headless','pyyaml'],'dependencies')
run_logged([PYTHON,'-c',"import torch,onnxruntime as ort; assert ort.__version__=='1.30.0'; print(torch.__version__,ort.__version__,'CPU-only evaluation')"],'versions')
READY=True


## 3. Tests, val-only data staging and fixed16 checks
Check exact model hashes first, then copy/hash all 3,769 validation images, labels and calibration files into the dedicated local folder. No train split is needed. Both real-image preprocessing and exact-input model/decoded parity must pass before full validation starts.

In [ ]:
run_logged([PYTHON,'-m','unittest','discover','-s','tests','-p','test_a2_lite_accuracy.py','-v'],'tests',MOBILE_REPO)
extra=[]
for candidate in CANDIDATES: extra += ['--candidate',candidate]
stage('prepare','--baseline',BASELINE,'--compact',COMPACT,'--dataset',DATASET,'--split-dir',SPLITS,*extra)


## 4. Paired inference and accuracy/nearby evaluation
This is much longer than a 16-image smoke test. Both graphs use the same images, calibration, decoder, top-50 selection and product evaluator. One atomic cache record holds each frame's paired predictions. If interrupted, rerun this section unchanged to verify/reuse completed frames. Failed accuracy gates are saved as findings, not treated as execution failures.

In [ ]:
stage('infer')
stage('evaluate')
REPORT=json.loads((OUTPUT/'a2_lite_accuracy_comparison.json').read_text())
assert REPORT['complete'] and REPORT['evaluated_images']==3769 and not REPORT['training_performed']
for role,result in REPORT['models'].items():
    print(role,json.dumps({'metrics':result['metrics'],'near_recall':{k:v['near_recall'] for k,v in result['nearby']['classes'].items()},'all_accuracy_gates_passed':result['all_accuracy_gates_passed']},indent=2))
print('640 minus full-resolution control:',json.dumps(REPORT['compact_minus_control_ap'],indent=2))
print('Return the ZIP for review before recovery training or distillation.')


## 5. Download the accuracy results
Run this even after an error so the full logs are included. Models, fixed fixtures and source images are excluded; paired predictions, cache identity, metrics and diagnostics are included.

In [ ]:
stage('bundle')
BUNDLE=OUTPUT/'a2_lite_accuracy_results.zip'
print(BUNDLE)
from google.colab import files
files.download(str(BUNDLE))
